# 06 — Test Two Decoding Enhancements (Flickr8k CNN–LSTM)

This Colab notebook evaluates **only** these two methods from [PR #11](https://github.com/JustinDang69/Automatic-Image-Captioning-with-CNN-and-LSTM-/pull/11), using the same saved CNN–LSTM checkpoint and frozen VGG16 image features:

1. **Top-k sampling:** `top_k=3`, fixed random `seed=42`
2. **Length-normalised beam search:** `beam_width=3`, `alpha=0.7`

It prints actual generated captions, displays Flickr8k images with reference captions, and calculates **corpus BLEU-1, BLEU-2, BLEU-3 and BLEU-4** for **both methods only**.

**Default BLEU check:** the first 20 *validation* images, with results shown immediately after the generation stage. Enable the optional full held-out test-set evaluation after your settings are final.

An executable cell below reproduces PR #11's `src/decoding_enhancements.py` **verbatim**, then connects it to the same caption decoder used by the project's baseline. No retraining is required. No results are fabricated or hard-coded.

## 1. Settings — fixed decoding hyperparameters

This notebook intentionally runs **only Top-k (k=3, seed 42)** and **Length-normalised Beam (width=3, alpha=0.7)**. Change the Google Drive paths if your trained model and feature files are saved somewhere else.

For a fast end-to-end check, `RUN_EXPLORATORY_BLEU=True` calculates scores on 20 validation images. This is a **preliminary** score, not the final held-out test result. Set `RUN_FINAL_TEST_BLEU=True` when ready for the full test evaluation.

In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/JustinDang69/Automatic-Image-Captioning-with-CNN-and-LSTM-.git"
REPO_BRANCH = "feature/pr11-colab-decoding-source"
DRIVE_PROJECT = Path("/content/drive/MyDrive/NIT3251_captioning")
RUN_NAME = "lstm_run1"

MODEL_DIR_OVERRIDE = None     # Folder with caption_model.keras and tokenizer/splits/metadata
FEATURES_DIR_OVERRIDE = None  # Folder with saved VGG16 features.npy and index files
DATA_DIR_OVERRIDE = None      # Folder containing Flickr8k Images/ and captions.txt

TOP_K = 3
BEAM_WIDTH = 3
ALPHA = 0.7
RANDOM_SEED = 42

EXAMPLE_SPLIT = "val"          # "val" recommended for inspecting/tuning, "test" for final examples
NUM_EXAMPLES = 3
VALIDATION_IMAGE_LIMIT = 20   # preliminary BLEU evaluation
RUN_EXPLORATORY_BLEU = True   # automatically show BLEU-1..4 for both methods on 20 val images
RUN_FINAL_TEST_BLEU = False   # set True for final full test split (usually 810 images)

assert TOP_K == 3 and BEAM_WIDTH == 3 and ALPHA == 0.7 and RANDOM_SEED == 42
assert EXAMPLE_SPLIT in {"val", "test"}
assert NUM_EXAMPLES > 0 and VALIDATION_IMAGE_LIMIT > 0

## 2. Mount Drive and download the right GitHub branch

The folder `/content/nit3251-enhancements` is deliberately separate from the existing `/content/nit3251-captioning` clone. This avoids accidentally importing older `main` versions of the decoding functions.

If you previously imported a different version of `src.caption_decoding` in the same session, use **Runtime → Restart session**, then run this notebook in order.

In [ ]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_ROOT = Path("/content/nit3251-enhancements")
    if not (PROJECT_ROOT / ".git").is_dir():
        subprocess.run(
            ["git", "clone", "--branch", REPO_BRANCH, REPO_URL, str(PROJECT_ROOT)],
            check=True,
        )
    else:
        for cmd in (
            ["fetch", "origin"],
            ["checkout", REPO_BRANCH],
            ["pull", "--ff-only"],
        ):
            subprocess.run(["git", "-C", str(PROJECT_ROOT), *cmd], check=True)
else:
    PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))
print("Working directory:", Path.cwd())
print("Git branch:", subprocess.check_output(
    ["git", "branch", "--show-current"], text=True
).strip())

## 3. Install packages and test PR #11's decoding implementation

The regression tests verify that the enhanced decoding functions work with the project's existing model interface. The actual trained-model captions are generated further down after the model and features are loaded.

In [ ]:
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r",
     str(PROJECT_ROOT / "requirements.txt")],
    check=True,
)
subprocess.run(
    [sys.executable, "-m", "pytest", "-q", "tests/test_decoding.py",
     "tests/test_decoding_regression.py", "tests/test_caption_decoding.py"],
    check=True,
)
print("Dependency installation and PR #11 decoding regression tests finished.")

## 4. PR #11's exact decoding-enhancement source code

The next executable cell is a verbatim copy of `src/decoding_enhancements.py` from PR #11. It defines `top_k_sampling`, `validate_top_k`, and `length_normalize_score`.

The following binding cell connects these notebook definitions to the imported caption-decoding functions. Both methods therefore use the code below, and changes here can be tested without modifying the source file in GitHub. Rerun the binding cell after editing the source cell.

### Exact PR #11 `src/decoding_enhancements.py`

In [ ]:
"""Decoding innovations (Abdul): length-normalised beam scores and Top-k sampling.

Both operate AFTER the softmax at inference time; the trained model is unchanged.
They are used by ``src.caption_decoding.beam_search_decode(alpha=...)`` and
``src.caption_decoding.greedy_decode(top_k=..., rng=...)``.
"""
import numpy as np

def length_normalize_score(raw_score, sequence_length, alpha=0.7):
    """
    Normalizes a sequence score to prevent bias against longer captions in Beam Search.

    Formula:
        normalized_score = raw_score / (sequence_length ^ alpha)

    Args:
        raw_score (float): The cumulative log-probability of the sequence.
        sequence_length (int): The number of tokens in the hypothesis. Beam search
            passes len(tokens), which includes startseq (and endseq once finished).
        alpha (float): The length penalty parameter (typically between 0.6 and 0.8).

    Returns:
        float: The length-normalized score.
    """
    safe_length = max(sequence_length, 1)  # Prevent division by zero
    return raw_score / (safe_length ** alpha)


def validate_top_k(top_k):
    """Raise ValueError unless ``top_k`` is a positive integer (k=1 behaves like greedy)."""
    if isinstance(top_k, bool) or not isinstance(top_k, (int, np.integer)) or top_k < 1:
        raise ValueError(f"top_k must be a positive integer (>= 1), got {top_k!r}")


# Parameter renamed from k to top_k to avoid confusion with the beam width k.
def top_k_sampling(probabilities, top_k=3, rng=None, banned_indices=None):
    """Sample the next token from the ``top_k`` most likely valid tokens.

    Args:
        probabilities: next-word probabilities, shape (vocab_size,) or (1, vocab_size).
        top_k: positive integer; how many of the most likely valid tokens to consider.
        rng: ``np.random.Generator``. Reuse one generator for a whole evaluation run
            (e.g. ``np.random.default_rng(42)``) so results are reproducible without
            reseeding at every token. None = a new unseeded generator.
        banned_indices: token IDs that must never be chosen (PAD, startseq, <unk>).

    Returns:
        int: the sampled token ID.

    Raises:
        ValueError: if ``top_k < 1``, or if no valid (non-banned) token has a
            positive probability - the same condition makes greedy decoding fail.
    """
    validate_top_k(top_k)
    if rng is None:
        rng = np.random.default_rng()

    probs_1d = np.asarray(probabilities).flatten().copy()

    # Zero out banned tokens (PAD, <unk>, startseq)
    if banned_indices:
        probs_1d[list(banned_indices)] = -1.0

    top_k_indices = np.argsort(probs_1d)[-top_k:]
    top_k_probs = probs_1d[top_k_indices]

    valid_mask = top_k_probs > 0
    top_k_indices = top_k_indices[valid_mask]
    top_k_probs = top_k_probs[valid_mask]

    if len(top_k_probs) == 0:
        # Every valid token has probability 0: there is nothing sensible to sample
        # (argmax would return an arbitrary zero-probability word).
        raise ValueError("No positive-probability valid next word for Top-k sampling")

    normalized_probs = top_k_probs / np.sum(top_k_probs)
    return int(rng.choice(top_k_indices, p=normalized_probs))


In [ ]:
import json, time, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display
from nltk.translate.bleu_score import corpus_bleu

# Use the existing Keras inference/padding implementations from PR #11.
import src.caption_decoding as caption_decoder_module
from src.caption_decoding import (
    load_assets, greedy_decode, beam_search_decode, _reference_tokens,
)
from src.data_preprocessing import get_dataset_paths

# Crucial: connect caption_decoding.py to the exact functions defined
# in the copied PR #11 source cell above. This allows you to edit those
# helper functions in the notebook and immediately test your changes.
caption_decoder_module.top_k_sampling = top_k_sampling
caption_decoder_module.length_normalize_score = length_normalize_score

assert caption_decoder_module.top_k_sampling is top_k_sampling
assert caption_decoder_module.length_normalize_score is length_normalize_score
print("Caption decoders now use the copied notebook functions.")

# Quick checks before loading the expensive trained model.
assert length_normalize_score(-5.0, 8, alpha=ALPHA) == -5.0 / (8 ** ALPHA)
assert top_k_sampling(
    np.array([0.8, 0.19, 0.01]), top_k=2,
    rng=np.random.default_rng(42), banned_indices={0}
) == 1
try:
    validate_top_k(0)
except ValueError:
    print("Invalid top_k correctly rejected.")
else:
    raise AssertionError("top_k=0 must be rejected")
try:
    top_k_sampling(np.array([1.0, 0.0]), top_k=2, banned_indices={0})
except ValueError:
    print("Missing valid next-word probability correctly rejected.")
else:
    raise AssertionError("No valid token should raise an error")


### Bind the copied code to caption generation

These checks verify `top_k=3` sampling and length-normalised beam scoring before loading the trained checkpoint.

## 5. Locate checkpoint, vocabulary and VGG16 features

**Required checkpoint folder contents**: `caption_model.keras` (or `best_caption_model.keras`), `tokenizer.json`, `splits.json`, and `metadata.json`.

**Required feature folder contents**: `features.npy`, `image_ids.json`, `progress.npy`, and `metadata.json`.

Your model weights and feature arrays are **not stored on GitHub**. If the files were generated by a teammate, copy or add a Drive shortcut to the right folder and adjust the overrides above. This cell gives actionable errors instead of attempting inference with missing files.

In [ ]:
MODEL_DIR = (Path(MODEL_DIR_OVERRIDE) if MODEL_DIR_OVERRIDE
             else DRIVE_PROJECT / "runs" / RUN_NAME)
FEATURES_DIR = (Path(FEATURES_DIR_OVERRIDE) if FEATURES_DIR_OVERRIDE
                else DRIVE_PROJECT / "features")
OUTPUT_DIR = DRIVE_PROJECT / "runs" / RUN_NAME / "decoding_enhancements"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

checkpoint_names = ["caption_model.keras", "best_caption_model.keras"]
checkpoint_found = next(
    (MODEL_DIR / f for f in checkpoint_names if (MODEL_DIR / f).is_file()), None
)
required_model_json = ["tokenizer.json", "splits.json", "metadata.json"]
required_features = ["features.npy", "image_ids.json", "progress.npy", "metadata.json"]
model_missing = [f for f in required_model_json if not (MODEL_DIR / f).is_file()]
features_missing = [f for f in required_features if not (FEATURES_DIR / f).is_file()]

print("MODEL_DIR:", MODEL_DIR)
print("Checkpoint found:", checkpoint_found)
print("Missing model metadata:", model_missing)
print("FEATURES_DIR:", FEATURES_DIR)
print("Missing features:", features_missing)

if checkpoint_found is None or model_missing:
    nearby = sorted(str(x) for x in DRIVE_PROJECT.glob("runs/*/*.keras")) if DRIVE_PROJECT.exists() else []
    raise FileNotFoundError(
        "Trained model or metadata is missing. Check MODEL_DIR_OVERRIDE / RUN_NAME. "
        "Nearby checkpoints: " + repr(nearby[:20]) +
        ". A trained .keras checkpoint cannot be recreated from GitHub code."
    )
if features_missing:
    raise FileNotFoundError(
        "Missing pre-extracted VGG16 files in FEATURES_DIR: " + repr(features_missing)
    )

## 6. Locate Flickr8k images and reference captions

If Flickr8k is already in Drive, the notebook uses it. Otherwise Colab downloads it through KaggleHub. Only **image files and captions** are downloaded; the pretrained LSTM checkpoint and saved VGG16 features must exist separately.

The same image IDs and saved test split from the training run are retained.

In [ ]:
if DATA_DIR_OVERRIDE:
    DATA_DIR = Path(DATA_DIR_OVERRIDE)
else:
    drive_data = DRIVE_PROJECT / "flickr8k"
    if (drive_data / "Images").is_dir() and (drive_data / "captions.txt").is_file():
        DATA_DIR = drive_data
    else:
        import kagglehub
        downloaded = Path(kagglehub.dataset_download("adityajn105/flickr8k"))
        candidates = [downloaded, *downloaded.iterdir()]
        DATA_DIR = next(
            (d for d in candidates
             if d.is_dir() and (d / "Images").is_dir()
             and (d / "captions.txt").is_file()),
            None,
        )
        if DATA_DIR is None:
            raise FileNotFoundError(
                f"Downloaded Flickr8k but could not find Images/ and captions.txt under {downloaded}"
            )
images_dir, captions_path = get_dataset_paths(DATA_DIR)
print("Flickr8k:", DATA_DIR, "| Images:", images_dir, "| Captions:", captions_path)

## 7. Load your trained CNN–LSTM checkpoint

The image encoder has already extracted the VGG16 feature vectors. `load_assets` reads the saved trained decoder, original tokenizer and image splits, and Flickr8k references. No weights are updated.

In [ ]:
assets = load_assets(
    model_dir=MODEL_DIR,
    features_dir=FEATURES_DIR,
    data_dir=DATA_DIR,
)
print("Model inputs:", assets.model.input_shape)
print("Model output:", assets.model.output_shape)
print("Max caption length:", assets.max_length)
print("Vocabulary entries:", assets.tokenizer.vocab_size)
print("Split sizes:", {name: len(ids) for name, ids in assets.splits.items()})
assert assets.model.output_shape[-1] == assets.tokenizer.vocab_size
assert len(assets.splits["test"]) > 0

## 8. Generate **only two** decoding outputs for each image

- **Top-k sampling (k=3, seed=42)**: stochastic next-word sampling among the three most likely allowed tokens, using a single seeded random generator throughout each run.
- **Length-normalised beam (width=3, α=0.7)**: searches up to three candidate sequences and ranks them with PR #11's length-normalisation formula.

The functions `greedy_decode` and `beam_search_decode` are reused as internal implementations, but the notebook **does not generate separate greedy or ordinary beam-search outputs**.

In [ ]:
METHODS = ("top_k", "length_norm")

def generate_two_methods(image_id, rng):
    if image_id not in assets.row_of:
        raise KeyError(f"VGG16 image feature not found: {image_id}")
    if image_id not in assets.references:
        raise KeyError(f"Reference captions not found: {image_id}")

    feature = assets.features[assets.row_of[image_id]]
    inputs = (assets.model, feature, assets.tokenizer, assets.max_length)

    start_time = time.perf_counter()
    top_k_caption = greedy_decode(*inputs, top_k=TOP_K, rng=rng)
    top_k_seconds = time.perf_counter() - start_time

    start_time = time.perf_counter()
    length_norm_caption = beam_search_decode(
        *inputs, beam_width=BEAM_WIDTH, alpha=ALPHA
    )
    length_norm_seconds = time.perf_counter() - start_time

    return {
        "image_id": image_id,
        "references": [" ".join(_reference_tokens(c))
                       for c in assets.references[image_id]],
        "captions": {
            "top_k": top_k_caption,
            "length_norm": length_norm_caption,
        },
        "seconds": {
            "top_k": top_k_seconds,
            "length_norm": length_norm_seconds,
        },
    }

## 9. Print actual trained-model caption examples

This section prints **only Top-k** and **Length-normalised Beam** predictions alongside a reference caption. A fixed seed makes the image selection and top-k sampling reproducible for a given execution order.

These examples can reveal missing objects, incorrect activities, hallucinations or unusually short captions.

In [ ]:
selected_ids = random.Random(RANDOM_SEED).sample(
    assets.splits[EXAMPLE_SPLIT],
    k=min(NUM_EXAMPLES, len(assets.splits[EXAMPLE_SPLIT])),
)
sampling_rng = np.random.default_rng(RANDOM_SEED)
examples = [generate_two_methods(img_id, sampling_rng) for img_id in selected_ids]

rows = []
for item in examples:
    print("\nIMAGE:", item["image_id"])
    print("Reference:", item["references"][0])
    print(f"Top-k (k={TOP_K}, seed={RANDOM_SEED}):", item["captions"]["top_k"])
    print(f"Length-normalised Beam (width={BEAM_WIDTH}, alpha={ALPHA}):",
          item["captions"]["length_norm"])
    rows.append({
        "Image": item["image_id"],
        "Reference": item["references"][0],
        f"Top-k (k={TOP_K}, seed={RANDOM_SEED})": item["captions"]["top_k"],
        f"Length-normalised Beam (w={BEAM_WIDTH}, alpha={ALPHA})":
            item["captions"]["length_norm"],
    })
display(pd.DataFrame(rows))

## 10. Display Flickr8k images and both generated captions

Each panel shows an actual Flickr8k image, all five human reference captions and the two generated captions. Results are also saved to your Google Drive as PNG/JSON for your report.

In [ ]:
import textwrap

fig, axes = plt.subplots(
    len(examples), 2,
    figsize=(16, max(4.2 * len(examples), 5)),
    squeeze=False,
    gridspec_kw={"width_ratios": [1, 1.7]},
)
for (left, right), item in zip(axes, examples):
    with Image.open(assets.images_dir / item["image_id"]) as img:
        left.imshow(img.convert("RGB"))
    left.set_title(item["image_id"], fontsize=10)
    left.axis("off")

    lines = ["Human references:"]
    lines.extend(f"{i + 1}. {caption}" for i, caption in enumerate(item["references"]))
    lines += [
        "",
        f"Top-k (k={TOP_K}, seed={RANDOM_SEED}): {item['captions']['top_k']}",
        f"Length-normalised Beam (width={BEAM_WIDTH}, alpha={ALPHA}): "
        f"{item['captions']['length_norm']}",
    ]
    right.text(
        0.0, 0.99,
        "\n".join(textwrap.fill(line, width=79, subsequent_indent="    ") for line in lines),
        va="top", transform=right.transAxes, fontsize=9, linespacing=1.35,
    )
    right.axis("off")

fig.tight_layout(h_pad=2)
figure_path = OUTPUT_DIR / f"examples_{EXAMPLE_SPLIT}_topk3_seed42_lnbeam3_alpha0.7.png"
fig.savefig(figure_path, dpi=160, bbox_inches="tight")
plt.show()
plt.close(fig)

json_path = OUTPUT_DIR / f"examples_{EXAMPLE_SPLIT}_topk3_seed42_lnbeam3_alpha0.7.json"
json_path.write_text(json.dumps({
    "split": EXAMPLE_SPLIT,
    "seed": RANDOM_SEED,
    "top_k": TOP_K,
    "beam_width": BEAM_WIDTH,
    "alpha": ALPHA,
    "examples": examples,
}, indent=2), encoding="utf-8")
print("Saved image:", figure_path)
print("Saved JSON:", json_path)

## 11. Calculate corpus BLEU-1, BLEU-2, BLEU-3 and BLEU-4

**Both methods only.** Each caption is compared against all five human references for the same Flickr8k image. The procedure uses the same caption cleaning and NLTK corpus BLEU definitions as the existing project.

For a quick test, the next section automatically calculates BLEU on **20 validation images**. Those numbers are exploratory. Section 13 can calculate final full-test scores after the hyperparameters have been fixed.

In [ ]:
BLEU_WEIGHTS = {
    "BLEU-1": (1., 0., 0., 0.),
    "BLEU-2": (.5, .5, 0., 0.),
    "BLEU-3": (1/3, 1/3, 1/3, 0.),
    "BLEU-4": (.25, .25, .25, .25),
}

def calculate_bleu_for_images(image_ids, split_name):
    if not image_ids:
        raise ValueError("Cannot evaluate an empty image set")
    references = []
    predictions = {method: [] for method in METHODS}
    duration = {method: [] for method in METHODS}
    rng = np.random.default_rng(RANDOM_SEED)  # one seeded RNG for entire split

    for index, image_id in enumerate(image_ids, start=1):
        item = generate_two_methods(image_id, rng)
        references.append([_reference_tokens(c) for c in assets.references[image_id]])
        for method in METHODS:
            predictions[method].append(item["captions"][method].split())
            duration[method].append(item["seconds"][method])
        if index % 100 == 0:
            print(f"{split_name}: captioned {index}/{len(image_ids)} images", flush=True)

    result_rows = []
    for method in METHODS:
        row = {
            "Method": (
                f"Top-k (k=3, seed=42)" if method == "top_k"
                else "Length-normalised Beam (w=3, alpha=0.7)"
            ),
            "Images": len(image_ids),
            "Mean seconds/caption": float(np.mean(duration[method])),
        }
        for name, weights in BLEU_WEIGHTS.items():
            row[name] = float(corpus_bleu(references, predictions[method], weights=weights))
        result_rows.append(row)

    return pd.DataFrame(result_rows)

## 12. Run a quick BLEU check on 20 validation images

**Enabled by default.** Running this cell displays a two-row BLEU table (Top-k and Length-normalised Beam) and saves its CSV file. It is a preliminary check, **not** the final 810-image test result.

In [ ]:
if RUN_EXPLORATORY_BLEU:
    validation_ids = assets.splits["val"][:VALIDATION_IMAGE_LIMIT]
    validation_scores = calculate_bleu_for_images(validation_ids, "validation")
    print(f"Exploratory BLEU on {len(validation_ids)} validation images:")
    display(validation_scores)
    validation_scores.to_csv(
        OUTPUT_DIR / "bleu_exploratory_validation_topk3_lnbeam3.csv", index=False
    )
else:
    print("Exploratory BLEU disabled. Set RUN_EXPLORATORY_BLEU=True to enable.")

## 13. Optional final BLEU comparison on all held-out test images

To obtain results appropriate for your report, set `RUN_FINAL_TEST_BLEU=True` in Section 1, then rerun this cell. Both methods use the **same test images and five references per image**. Keep `k=3`, `seed=42`, `width=3`, and `alpha=0.7` fixed.

The full evaluation may take substantial time. The notebook writes a two-row CSV, JSON with the exact parameters and a chart.

In [ ]:
if RUN_FINAL_TEST_BLEU:
    test_ids = assets.splits["test"]
    full_scores = calculate_bleu_for_images(test_ids, "test")
    print(f"Final corpus BLEU results on {len(test_ids)} test images:")
    display(full_scores)
    full_scores.to_csv(OUTPUT_DIR / "bleu_full_test_topk3_lnbeam3.csv", index=False)
    summary = {
        "split": "test",
        "images": len(test_ids),
        "model_dir": str(MODEL_DIR),
        "features_dir": str(FEATURES_DIR),
        "random_seed": RANDOM_SEED,
        "top_k": TOP_K,
        "beam_width": BEAM_WIDTH,
        "alpha": ALPHA,
        "results": full_scores.to_dict(orient="records"),
    }
    (OUTPUT_DIR / "bleu_full_test_topk3_lnbeam3.json").write_text(
        json.dumps(summary, indent=2), encoding="utf-8"
    )
    print("Results saved to:", OUTPUT_DIR)
else:
    print("Full test evaluation disabled. Set RUN_FINAL_TEST_BLEU=True to enable.")

## 14. Plot only the two calculated BLEU result series

This graph uses the BLEU values calculated in the notebook, not hardcoded examples. It prefers final test scores when available; otherwise it displays the quick validation scores.

In [ ]:
if "full_scores" in globals() and RUN_FINAL_TEST_BLEU:
    scores_to_plot = full_scores
    chart_split = "test"
elif "validation_scores" in globals() and RUN_EXPLORATORY_BLEU:
    scores_to_plot = validation_scores
    chart_split = "exploratory_validation"
else:
    scores_to_plot = None

if scores_to_plot is not None:
    plot_data = scores_to_plot.set_index("Method")[list(BLEU_WEIGHTS)].T
    ax = plot_data.plot(kind="bar", figsize=(10, 5), rot=0)
    ax.set_title(f"Top-k vs Length-normalised Beam: {chart_split.replace('_', ' ')} BLEU")
    ax.set_ylabel("Corpus BLEU score")
    ax.set_xlabel("Metric")
    ax.grid(axis="y", alpha=0.2)
    ax.figure.tight_layout()
    chart_path = OUTPUT_DIR / f"bleu_topk3_lnbeam3_{chart_split}.png"
    ax.figure.savefig(chart_path, dpi=180)
    plt.show()
    plt.close(ax.figure)
    print("Saved chart:", chart_path)
else:
    print("No BLEU scores computed yet.")

## 15. Reading and reporting the two results

- **Top-k (k=3, seed=42):** Samples one of the three most likely valid tokens at each step. A fixed random seed makes a full run reproducible, assuming the same image order and environment. Sampling can add variety but may reduce BLEU.
- **Length-normalised beam (width=3, alpha=0.7):** Retains three candidate sequences and divides each cumulative log score by the sequence length to the power `0.7`. PR #11 counts `startseq` and `endseq` when present in the sequence length.
- **BLEU-1 to BLEU-4:** Measures n-gram overlap with all five reference captions per image. Compare the two rows in Section 12 or 13, but inspect actual image-caption examples to detect hallucinations or omitted objects.
- **Experiment protocol:** The automatic quick BLEU numbers are computed on only 20 validation images and must be labelled **preliminary**. For final reporting, turn on `RUN_FINAL_TEST_BLEU` and use the saved held-out test split (usually 810 images).

This notebook produces **no separate Greedy or Standard Beam baseline predictions**. Both improved methods use the previously trained decoder without retraining VGG16 or LSTM.